<img src="https://theaiengineer.dev/tae_logo_gw_flatter.png" width="35%" align="right">


[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yhilpisch/colab/blob/main/notebooks/08_stable_diffusion_gpu.ipynb)


# colab — Text-to-Image Generation on a Free GPU

## _Running Stable Diffusion Interactively with a T4 GPU_

**&copy; Dr. Yves J. Hilpisch**<br>AI-Powered by Different LLMs.

## How to Use This Notebook

- **Goal**: Generate images from text prompts using Stable Diffusion on a free Colab
    GPU.
- **Hardware**: Designed for Google Colab (T4 default). GPU is auto-detected.
- **Model**: `runwayml/stable-diffusion-v1-5` in half-precision (~4 GB) fits
    comfortably on a T4.
- **Why GPU?**: A single 512×512 image takes minutes on CPU but seconds on a T4.
- **GPU Reference**: See `notebooks/colab_gpus.md` for the full GPU comparison table.

### Roadmap

1. **Environment**: Check GPU and install `diffusers`, `transformers`, `accelerate`.
2. **Load Model**: Download the Stable Diffusion pipeline in `float16`.
3. **Generate**: Create images from prompts with varying inference steps.
4. **Negative Prompts**: Use negative prompts to improve quality.
5. **Benchmark**: Measure generation time vs step count.

### Environment Setup

Verify the GPU and install the libraries needed for Stable Diffusion inference.

In [ ]:
#@title Check GPU and install dependencies
!nvidia-smi
!pip -q install diffusers transformers accelerate
!pip -q install matplotlib

In [ ]:
#@title Imports
import time
import torch
from diffusers import StableDiffusionPipeline
import matplotlib.pyplot as plt
plt.style.use("seaborn-v0_8")

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"PyTorch device: {device}")

### Load the Stable Diffusion Pipeline

We download `runwayml/stable-diffusion-v1-5` in `float16` and move it to the GPU.
The T4 has 16 GB VRAM, so ~4 GB for the model leaves plenty of room for the latents.

In [ ]:
#@title Download and load the model
MODEL_ID = "runwayml/stable-diffusion-v1-5"

pipe = StableDiffusionPipeline.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    safety_checker=None,
    requires_safety_checker=False,
).to("cuda")

print("Pipeline loaded.")
print(f"VRAM allocated: {torch.cuda.memory_allocated() / 1e9:.2f} GB")

### Basic Text-to-Image Generation

A simple helper that generates an image from a prompt and displays it inline.

In [ ]:
#@title Generation helper
def generate(prompt, steps=25, guidance=7.5, neg_prompt="",
             seed=42):
    """Generate an image and return it with wall time."""
    generator = torch.Generator("cuda").manual_seed(seed)
    start = time.perf_counter()
    image = pipe(
        prompt,
        num_inference_steps=steps,
        guidance_scale=guidance,
        negative_prompt=neg_prompt,
        generator=generator,
    ).images[0]
    elapsed = time.perf_counter() - start
    return image, elapsed

In [ ]:
#@title First image
PROMPT = (
    "a photograph of an astronaut riding a horse, "
    "digital art, highly detailed"
)
img, t = generate(PROMPT, steps=25)
print(f"Generated in {t:.2f} s")
img

### Effect of Inference Steps

Fewer steps = faster generation, but often lower quality.
We generate the same prompt with 5, 10, and 25 steps side by side.

In [ ]:
#@title Compare step counts
step_counts = [5, 10, 25]
fig, axes = plt.subplots(1, len(step_counts), figsize=(12, 4))

for ax, steps in zip(axes, step_counts):
    img, t = generate(PROMPT, steps=steps, seed=42)
    ax.imshow(img)
    ax.set_title(f"{steps} steps\n{t:.1f} s")
    ax.axis("off")

plt.tight_layout()
plt.show()

### Using Negative Prompts

Negative prompts tell the model what to avoid.
This is a simple but powerful technique to improve coherence and reduce artifacts.

In [ ]:
#@title With and without negative prompt
NEG = "blurry, low quality, distorted, deformed, ugly"

img_no_neg, t1 = generate(PROMPT, steps=25, seed=123)
img_neg, t2 = generate(PROMPT, steps=25, neg_prompt=NEG, seed=123)

fig, axes = plt.subplots(1, 2, figsize=(10, 5))
axes[0].imshow(img_no_neg)
axes[0].set_title(f"No negative prompt\n{t1:.1f} s")
axes[0].axis("off")
axes[1].imshow(img_neg)
axes[1].set_title(f"With negative prompt\n{t2:.1f} s")
axes[1].axis("off")
plt.tight_layout()
plt.show()

### Benchmark: Steps vs Time

We run a fixed prompt across a range of step counts and plot the trade-off.

In [ ]:
#@title Step-count benchmark
step_range = [5, 10, 15, 20, 25, 30, 50]
times = []

for s in step_range:
    _, t = generate(PROMPT, steps=s, seed=42)
    times.append(t)
    print(f"Steps {s:3d}: {t:.2f} s")

plt.figure(figsize=(6, 4))
plt.plot(step_range, times, marker="o")
plt.xlabel("Inference steps")
plt.ylabel("Wall time (seconds)")
plt.title("Generation time vs step count")
plt.grid(True)
plt.show()

### Swapping Models

Because a T4 still has ~16 GB of VRAM, you can try other lightweight models.
A few drop-in replacements (change `MODEL_ID` above):

| Model | Size (fp16) | Notes |
|-------|-------------|-------|
| `runwayml/stable-diffusion-v1-5` | ~4 GB | Default in this notebook |
| `CompVis/stable-diffusion-v1-4` | ~4 GB | Earlier version, similar quality |
| `segmind/SSD-1B` | ~3 GB | Smaller, faster, slightly lower fidelity |
| `stabilityai/stable-diffusion-2-1` | ~4 GB | Improved quality, 768×768 native |

For larger models (e.g. SDXL) that do not fit in native 16-bit,
use sequential CPU offloading:
```python
pipe.enable_sequential_cpu_offload()
```

### Exercises

1. **Different prompt**: Try `"a futuristic city at sunset, cyberpunk, "
   `"8k, concept art"` and compare 10 vs 50 steps.
2. **Guidance scale**: Fix 25 steps and vary `guidance_scale` from 3 to 15.
   Observe how strongly the image follows the prompt.
3. **Resolution**: Change `height` and `width` to 768×768.
   Does the T4 run out of VRAM?
4. **Batch generation**: Generate 4 images at once with a batch prompt list
   and compare throughput (images per minute).
5. **Another model**: Swap in `stabilityai/stable-diffusion-2-1` and compare
   the same prompt at identical settings.

<img src="https://theaiengineer.dev/tae_logo_gw_flatter.png" width="35%" align="right">
